In [1]:
import os
import requests
from dotenv import load_dotenv

load_dotenv()

OPENAQ_BASE_URL = "https://api.openaq.org/v3"
DELHI_COORDINATES = "28.6139,77.2090"  # lat,lon
DELHI_RADIUS_M = 25000  # v3 max radius

headers = {"X-API-Key": os.getenv("OPENAQ_API_KEY")}
print("Key loaded:", headers["X-API-Key"] is not None)

Key loaded: True


In [2]:
params = {
    "coordinates": DELHI_COORDINATES,
    "radius": DELHI_RADIUS_M,
    "limit": 100,
}

response = requests.get(f"{OPENAQ_BASE_URL}/locations", headers=headers, params=params)
response.raise_for_status()

locations = response.json()["results"]
print(f"Found {len(locations)} stations near Delhi")

Found 96 stations near Delhi


In [3]:
import pandas as pd

station_rows = []
for loc in locations:
    station_rows.append({
        "id": loc["id"],
        "name": loc["name"],
        "lat": loc["coordinates"]["latitude"],
        "lon": loc["coordinates"]["longitude"],
        "parameters": ", ".join(s["parameter"]["name"] for s in loc["sensors"]),
    })

stations_df = pd.DataFrame(station_rows)
stations_df

,id,name,lat,lon,parameters
0,13,"Delhi Technological University, Delhi - CPCB",28.744000,77.120000,"no2, o3, pm25"
1,15,IGI Airport,28.560000,77.094000,"co, no2, o3, pm10, pm25"
2,16,Civil Lines,28.678700,77.226200,"co, no2, pm10, pm25"
3,17,"R K Puram, Delhi - DPCC",28.563262,77.186937,"co, co, no, no2, no2, nox, o3, o3, pm10, pm10,..."
4,50,"Punjabi Bagh, Delhi - DPCC",28.674045,77.131023,"co, co, no, no2, no2, nox, o3, o3, pm10, pm10,..."
...,...,...,...,...,...
91,6254665,"Commonwealth Sports Complex, Delhi - DPCC",28.615828,77.271992,"co, no, no2, nox, o3, pm10, pm25, relativehumi..."
92,6254666,"IGNOU_Maidan Garhi, Delhi - DPCC",28.493624,77.201159,"co, no, no2, nox, o3, pm10, pm25, relativehumi..."
93,6257818,"Cantonment Area, Delhi - DPCC",28.594169,77.125100,"co, no, no2, nox, o3, pm10, pm25, relativehumi..."
94,6299494,"Prashant Garden, Khora - UPPCB",28.611190,77.342060,"co, no, no2, nox, o3, pm10, pm25, relativehumi..."


In [4]:
import os

os.makedirs("../data", exist_ok=True)
stations_df.to_csv("../data/delhi_stations.csv", index=False)
print("Saved", len(stations_df), "stations")

Saved 96 stations


In [5]:
sample_location = locations[0]
response = requests.get(
    f"{OPENAQ_BASE_URL}/locations/{sample_location['id']}/latest",
    headers=headers,
)
response.raise_for_status()
latest_readings = response.json()["results"]
latest_readings


[{'datetime': {'utc': '2018-02-22T04:00:00Z',
   'local': '2018-02-22T09:30:00+05:30'},
  'value': 40.3,
  'coordinates': {'latitude': 28.744, 'longitude': 77.12},
  'sensorsId': 13866,
  'locationsId': 13},
 {'datetime': {'utc': '2018-02-22T04:00:00Z',
   'local': '2018-02-22T09:30:00+05:30'},
  'value': 451.0,
  'coordinates': {'latitude': 28.744, 'longitude': 77.12},
  'sensorsId': 13864,
  'locationsId': 13}]